# Feature Engineering — No-Leak Competitive Markets

This notebook creates the first model-ready dataset for the dissertation.

**Financial baseline features:** `Lag1_Return`, `Lag2_Return`, `Lag3_Return`.

**No technical indicators** are added.

**Polymarket features:** lagged probability, one-day probability change, change from market start, time to resolution, and a probability × time interaction.

Future outcome fields such as `Forward_Log_Return`, `Price_Next`, and `Winner` are kept only for target/evaluation and must never be used as predictors.


## Step 1 — Upload the Three Required Excel Files

Upload:
1. `Final_Competitive_Markets_BASE_Modeling_Dataset.xlsx`
2. `Final Polymarkets datasets(1).xlsx`
3. `Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx`


## Step 2 — Import Libraries and Load the Three Files


In [ ]:
# ============================================================
# STEP 2: Import Libraries and Load Data
# ============================================================
import pandas as pd
import numpy as np

BASE_PATH = "/content/Final_Competitive_Markets_BASE_Modeling_Dataset.xlsx"
POLY_PATH = "/content/Final Polymarkets datasets.xlsx"
FIN_PATH = "/content/Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx"

base = pd.read_excel(BASE_PATH, sheet_name="Master_Long")
poly = pd.read_excel(POLY_PATH, sheet_name="All Daily Wide")
finance = pd.read_excel(FIN_PATH, sheet_name="All Financial Data")

for df in [base, poly, finance]:
    df["Date"] = pd.to_datetime(df["Date"])
base["Next_Financial_Date"] = pd.to_datetime(base["Next_Financial_Date"])

print("BASE:", base.shape)
print("Polymarket:", poly.shape)
print("Yahoo Finance:", finance.shape)
print("\nMarkets:")
print(base.groupby("Market_ID")["Event_ID"].nunique())
display(base.head())


## Step 3 — Validate the BASE Dataset


In [ ]:
# ============================================================
# STEP 3: Validate BASE Dataset
# ============================================================
print("Total Event IDs:", base["Event_ID"].nunique())
print("\nWinner counts per event:")
print(base.groupby("Event_ID")["Winner"].sum().value_counts().sort_index())
print("\nChoice-set sizes:")
print(base.groupby("Event_ID")["Asset"].nunique().value_counts().sort_index())
print("\nDuplicate Event_ID + Asset rows:", base.duplicated(["Event_ID","Asset"]).sum())


## Step 4 — Prepare Yahoo Finance Prices

Only adjusted close prices are used to calculate historical log returns. No technical indicators are created.


In [ ]:
# ============================================================
# STEP 4: Prepare Yahoo Finance Prices
# ============================================================
prices = finance[["Date","Asset","Adj Close"]].copy()
prices = prices.rename(columns={"Adj Close":"Price"})
price_wide = prices.pivot(index="Date", columns="Asset", values="Price").sort_index()
print(price_wide.notna().sum())
display(price_wide.head())


## Step 5 — Create Lag1, Lag2, and Lag3 Financial Returns

For an event on date **t**:
- `Lag1_Return`: previous common trading date → t
- `Lag2_Return`: one common trading period earlier
- `Lag3_Return`: two common trading periods earlier

The target remains t → next common financial date.


In [ ]:
# ============================================================
# STEP 5A: Function for Historical Return Lags
# ============================================================
def build_financial_lags(price_wide, assets, market_id):
    common = price_wide[assets].dropna().copy()
    common["Financial_Prev1_Date"] = common.index.to_series().shift(1)
    common["Financial_Prev2_Date"] = common.index.to_series().shift(2)
    common["Financial_Prev3_Date"] = common.index.to_series().shift(3)
    out=[]
    for asset in assets:
        p = common[asset]
        temp = pd.DataFrame({
            "Date": common.index,
            "Market_ID": market_id,
            "Asset": asset,
            "Lag1_Return": np.log(p / p.shift(1)).values,
            "Lag2_Return": np.log(p.shift(1) / p.shift(2)).values,
            "Lag3_Return": np.log(p.shift(2) / p.shift(3)).values,
            "Financial_Prev1_Date": common["Financial_Prev1_Date"].values,
            "Financial_Prev2_Date": common["Financial_Prev2_Date"].values,
            "Financial_Prev3_Date": common["Financial_Prev3_Date"].values
        })
        out.append(temp)
    return pd.concat(out, ignore_index=True)


In [ ]:
# ============================================================
# STEP 5B: Build Lags for Each Competitive Market
# ============================================================
fin_bg25 = build_financial_lags(price_wide,["Bitcoin","Gold"],"BTC_GOLD_2025")
fin_bg26 = build_financial_lags(price_wide,["Bitcoin","Gold"],"BTC_GOLD_2026")
fin_three = build_financial_lags(price_wide,["Bitcoin","Gold","SP500"],"BTC_GOLD_SP500")
financial_features = pd.concat([fin_bg25,fin_bg26,fin_three], ignore_index=True)
print("Financial feature rows:", len(financial_features))
display(financial_features.head(10))


## Step 6 — Convert Raw Polymarket Data to a Full Daily Long Panel

This uses the full daily Polymarket calendar, including weekends, so the one-day probability change is really a one-day change.


In [ ]:
# ============================================================
# STEP 6: Build Daily Long Polymarket Panel
# ============================================================
poly_rows=[]

# BTC vs Gold 2025
t = poly[["Date","BTC_Outperform_Gold_2025_Prob"]].dropna()
for _,r in t.iterrows():
    p=r["BTC_Outperform_Gold_2025_Prob"]
    poly_rows += [
        {"Market_ID":"BTC_GOLD_2025","Date":r["Date"],"Asset":"Bitcoin","PM_Prob_Daily":p},
        {"Market_ID":"BTC_GOLD_2025","Date":r["Date"],"Asset":"Gold","PM_Prob_Daily":1-p}
    ]

# BTC vs Gold 2026
t = poly[["Date","BTC_Outperform_Gold_2026_Prob"]].dropna()
for _,r in t.iterrows():
    p=r["BTC_Outperform_Gold_2026_Prob"]
    poly_rows += [
        {"Market_ID":"BTC_GOLD_2026","Date":r["Date"],"Asset":"Bitcoin","PM_Prob_Daily":p},
        {"Market_ID":"BTC_GOLD_2026","Date":r["Date"],"Asset":"Gold","PM_Prob_Daily":1-p}
    ]

# BTC vs Gold vs SP500
t = poly[["Date","Bitcoin_Best_2026_Prob_Norm","Gold_Best_2026_Prob_Norm","SP500_Best_2026_Prob_Norm"]].dropna()
for _,r in t.iterrows():
    poly_rows += [
        {"Market_ID":"BTC_GOLD_SP500","Date":r["Date"],"Asset":"Bitcoin","PM_Prob_Daily":r["Bitcoin_Best_2026_Prob_Norm"]},
        {"Market_ID":"BTC_GOLD_SP500","Date":r["Date"],"Asset":"Gold","PM_Prob_Daily":r["Gold_Best_2026_Prob_Norm"]},
        {"Market_ID":"BTC_GOLD_SP500","Date":r["Date"],"Asset":"SP500","PM_Prob_Daily":r["SP500_Best_2026_Prob_Norm"]}
    ]

poly_daily_long = pd.DataFrame(poly_rows).sort_values(["Market_ID","Asset","Date"]).reset_index(drop=True)
print("Daily Polymarket rows:", len(poly_daily_long))
display(poly_daily_long.head(12))


## Step 7 — Create Leakage-Safe Polymarket Features

The main specification uses the **previous available Polymarket day** so same-day timing cannot leak into the forward target.


In [ ]:
# ============================================================
# STEP 7: Create Lagged Polymarket Features
# ============================================================
g = poly_daily_long.groupby(["Market_ID","Asset"], group_keys=False)

poly_daily_long["PM_Signal_Date_Lag1"] = g["Date"].shift(1)
poly_daily_long["PM_Prob_Lag1"] = g["PM_Prob_Daily"].shift(1)
poly_daily_long["PM_Prob_Lag2_Audit"] = g["PM_Prob_Daily"].shift(2)
poly_daily_long["PM_Prob_Lag3_Audit"] = g["PM_Prob_Daily"].shift(3)

# Change that was fully known before event date t
poly_daily_long["PM_Change_1D_Lag1"] = (
    poly_daily_long["PM_Prob_Lag1"] - poly_daily_long["PM_Prob_Lag2_Audit"]
)

poly_daily_long["PM_Start_Prob"] = g["PM_Prob_Daily"].transform("first")
poly_daily_long["PM_Change_From_Start_Lag1"] = (
    poly_daily_long["PM_Prob_Lag1"] - poly_daily_long["PM_Start_Prob"]
)

display(poly_daily_long.head(15))


## Step 8 — Add Time to Resolution

Default resolution dates are set below. If the official Polymarket contract page gives a different exact resolution date, change the dictionary before modelling.


In [ ]:
# ============================================================
# STEP 8: Add Time-to-Resolution Features
# ============================================================
RESOLUTION_DATES = {
    "BTC_GOLD_2025": pd.Timestamp("2026-01-01"),
    "BTC_GOLD_2026": pd.Timestamp("2027-01-01"),
    "BTC_GOLD_SP500": pd.Timestamp("2027-01-01")
}

poly_daily_long["Resolution_Date"] = poly_daily_long["Market_ID"].map(RESOLUTION_DATES)
poly_daily_long["Days_to_Resolution"] = (
    poly_daily_long["Resolution_Date"] - poly_daily_long["PM_Signal_Date_Lag1"]
).dt.days
poly_daily_long["Days_to_Resolution_Years"] = poly_daily_long["Days_to_Resolution"] / 365.25
poly_daily_long["PM_Prob_x_TimeToResolution"] = (
    poly_daily_long["PM_Prob_Lag1"] * poly_daily_long["Days_to_Resolution_Years"]
)

display(poly_daily_long[[
    "Market_ID","Date","Asset","PM_Signal_Date_Lag1","PM_Prob_Lag1",
    "Days_to_Resolution","Days_to_Resolution_Years","PM_Prob_x_TimeToResolution"
]].dropna().head(10))


## Step 9 — Select Polymarket Features for the Master Dataset


In [ ]:
# ============================================================
# STEP 9: Select Polymarket Features
# ============================================================
poly_features = poly_daily_long[[
    "Market_ID","Date","Asset","PM_Signal_Date_Lag1","PM_Prob_Lag1",
    "PM_Change_1D_Lag1","PM_Change_From_Start_Lag1",
    "Days_to_Resolution","Days_to_Resolution_Years","PM_Prob_x_TimeToResolution"
]].copy()
print("Polymarket feature rows:", len(poly_features))


## Step 10 — Merge Safe Features into the BASE Dataset

The same-day Polymarket probability is renamed `PM_Prob_Current_Audit` so it is not accidentally used in the main no-leak models.


In [ ]:
# ============================================================
# STEP 10: Merge Financial and Polymarket Features
# ============================================================
model_data = base.rename(columns={"PM_Prob":"PM_Prob_Current_Audit"}).copy()
model_data = model_data.merge(financial_features, on=["Market_ID","Date","Asset"], how="left")
model_data = model_data.merge(poly_features, on=["Market_ID","Date","Asset"], how="left")
print("Merged shape:", model_data.shape)
display(model_data.head(12))


## Step 11 — Define Baseline and Enhanced Predictor Sets

### Baseline
`Lag1_Return`, `Lag2_Return`, `Lag3_Return`

### Enhanced
The same return lags plus lagged Polymarket information.

No technical indicators are used.


In [ ]:
# ============================================================
# STEP 11: Define Predictor Sets
# ============================================================
BASELINE_FEATURES = [
    "Lag1_Return","Lag2_Return","Lag3_Return"
]

ENHANCED_FEATURES = [
    "Lag1_Return","Lag2_Return","Lag3_Return",
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Change_From_Start_Lag1",
    "PM_Prob_x_TimeToResolution"
]

ML_EXTRA_CONTEXT = ["Days_to_Resolution_Years"]

TARGET_COLUMNS = ["Price_Next","Forward_Log_Return","Winner"]
AUDIT_ONLY_COLUMNS = [
    "PM_Prob_Current_Audit","Price_t","Next_Financial_Date",
    "PM_Signal_Date_Lag1","Financial_Prev1_Date",
    "Financial_Prev2_Date","Financial_Prev3_Date"
]

print("BASELINE:", BASELINE_FEATURES)
print("\nENHANCED:", ENHANCED_FEATURES)
print("\nTARGET/OUTCOME — NEVER USE AS PREDICTORS:", TARGET_COLUMNS)


## Step 12 — Keep Only Complete Events

The first few events can be missing Lag2/Lag3 or lagged Polymarket history. An **entire Event ID** is removed if any alternative is missing a required predictor.


In [ ]:
# ============================================================
# STEP 12: Keep Only Complete Events
# ============================================================
required_features = list(dict.fromkeys(BASELINE_FEATURES + ENHANCED_FEATURES))
row_complete = model_data[required_features].notna().all(axis=1)
event_complete = row_complete.groupby(model_data["Event_ID"]).all()
valid_event_ids = event_complete[event_complete].index

model_ready = model_data[model_data["Event_ID"].isin(valid_event_ids)].copy()
model_ready = model_ready.sort_values(["Date","Market_ID","Event_ID","Asset"]).reset_index(drop=True)

print("Events before:", model_data["Event_ID"].nunique())
print("Events after:", model_ready["Event_ID"].nunique())
print("\nEvents by market:")
print(model_ready.groupby("Market_ID")["Event_ID"].nunique())


## Step 13 — Leakage and Structure Checks


In [ ]:
# ============================================================
# STEP 13: Leakage and Structural Validation
# ============================================================
print("="*75)
print("LEAKAGE-SAFETY CHECK")
print("="*75)

# Polymarket signal strictly before event date
pm_bad = model_ready[model_ready["PM_Signal_Date_Lag1"] >= model_ready["Date"]]
print("PM signal date on/after event date:", len(pm_bad))

# Previous financial date strictly before event date
fin_bad = model_ready[model_ready["Financial_Prev1_Date"] >= model_ready["Date"]]
print("Financial Prev1 date on/after event date:", len(fin_bad))

print("\nWinner counts:")
print(model_ready.groupby("Event_ID")["Winner"].sum().value_counts().sort_index())

print("\nChoice-set sizes:")
print(model_ready.groupby("Event_ID")["Asset"].nunique().value_counts().sort_index())

print("\nMissing required predictors:")
print(model_ready[required_features].isna().sum())

lagged_prob_sums = model_ready.groupby("Event_ID")["PM_Prob_Lag1"].sum()
print("\nMax lagged probability-sum deviation from 1:", (lagged_prob_sums-1).abs().max())


## Step 14 — Inspect the Final Model-Ready Dataset


In [ ]:
# ============================================================
# STEP 14: Inspect Final Dataset
# ============================================================
display_columns = [
    "Market_ID","Event_ID","Date","Asset",
    "Lag1_Return","Lag2_Return","Lag3_Return",
    "PM_Prob_Lag1","PM_Change_1D_Lag1","PM_Change_From_Start_Lag1",
    "Days_to_Resolution_Years","PM_Prob_x_TimeToResolution",
    "Forward_Log_Return","Winner"
]
display(model_ready[display_columns].head(20))


## Step 15 — Final Event Summary


In [ ]:
# ============================================================
# STEP 15: Final Event Summary
# ============================================================
event_summary = model_ready.groupby("Market_ID").agg(
    Events=("Event_ID","nunique"),
    Rows=("Event_ID","size"),
    First_Date=("Date","min"),
    Last_Date=("Date","max")
)
display(event_summary)
print("Total Event IDs:", model_ready["Event_ID"].nunique())
print("Total rows:", len(model_ready))


## Step 16 — Save the Final Model-Ready Excel File


In [ ]:
# ============================================================
# STEP 16: Save Final Model-Ready Dataset
# ============================================================
output_path = "/content/Final_Competitive_Markets_MODEL_READY_Dataset.xlsx"

feature_guide = pd.DataFrame({
    "Feature":[
        "Lag1_Return","Lag2_Return","Lag3_Return",
        "PM_Prob_Lag1","PM_Change_1D_Lag1","PM_Change_From_Start_Lag1",
        "Days_to_Resolution_Years","PM_Prob_x_TimeToResolution",
        "Forward_Log_Return","Winner"
    ],
    "Role":[
        "Baseline predictor","Baseline predictor","Baseline predictor",
        "Enhanced Polymarket predictor","Enhanced Polymarket predictor",
        "Enhanced Polymarket predictor","ML/context feature",
        "Enhanced interaction predictor","Outcome - DO NOT USE AS PREDICTOR","Target"
    ]
})

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    model_ready.to_excel(writer, sheet_name="Model_Ready_Master", index=False)
    for market_id, sheet_name in {
        "BTC_GOLD_2025":"BTC_Gold_2025",
        "BTC_GOLD_2026":"BTC_Gold_2026",
        "BTC_GOLD_SP500":"BTC_Gold_SP500"
    }.items():
        model_ready[model_ready["Market_ID"]==market_id].to_excel(writer, sheet_name=sheet_name, index=False)
    event_summary.to_excel(writer, sheet_name="Event_Summary")
    feature_guide.to_excel(writer, sheet_name="Feature_Guide", index=False)

print("Saved:", output_path)


## Step 17 — Download the Final Model-Ready Dataset


In [ ]:
# ============================================================
# STEP 17: Download the Final Dataset
# ============================================================
from google.colab import files
files.download("/content/Final_Competitive_Markets_MODEL_READY_Dataset.xlsx")
